# 🌊 JalRakshak Health AI - Outbreak Risk Prediction & EDA

This notebook demonstrates the exploratory data analysis (EDA), statistical visualization, and machine learning pipeline for **JalRakshak Health AI** - a smart surveillance and early warning system designed for waterborne outbreaks in rural areas.

### Core ML Pipeline
1. **Exploratory Data Analysis (EDA)**: Examine features and relationship to outbreak status.
2. **Visualization**: Plot feature distributions and create a Pearson correlation heatmap.
3. **Model Selection & Comparison**: Train and compare `RandomForestClassifier` and `XGBClassifier` (XGBoost).
4. **Model Export**: Serialize the best-performing model (XGBoost) to `pickle` format for deployment on the Flask API server.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Dataset Loading / Generation

We will load the surveillance dataset from `../data/dataset.csv` or generate a highly realistic dataset based on symptom columns (Fever, Diarrhea, Vomiting) and water contamination records (`water_numeric`), modeling standard epidemiological dynamics for waterborne outbreaks (e.g. Cholera, Typhoid, Gastroenteritis).

In [ ]:
dataset_path = '../data/dataset.csv'

if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    print(f"Successfully loaded existing dataset from {dataset_path}")
else:
    print("Generating new synthetic surveillance dataset...")
    np.random.seed(42)
    num_samples = 2500

    # Feature distribution probabilities
    fever = np.random.choice([0, 1], size=num_samples, p=[0.7, 0.3])
    diarrhea = np.random.choice([0, 1], size=num_samples, p=[0.75, 0.25])
    vomiting = np.random.choice([0, 1], size=num_samples, p=[0.8, 0.20])
    water_numeric = np.random.choice([0, 1], size=num_samples, p=[0.6, 0.4])

    outbreak = []
    for f, d, v, w in zip(fever, diarrhea, vomiting, water_numeric):
        prob = 0.05  # Base background rate
        
        # Water contamination dramatically increases likelihood
        if w == 1:
            prob += 0.20
            if d == 1:  # Dirty water + Diarrhea is highly suggestive of Cholera/dysentery
                prob += 0.40
            if v == 1:
                prob += 0.15
            if f == 1:
                prob += 0.10
        else:
            # Clean water, but clustered symptoms could still indicate local transmission
            if d == 1 and v == 1:
                prob += 0.30
            if f == 1 and d == 1:
                prob += 0.20
                
        prob = min(max(prob, 0.02), 0.95)
        label = np.random.binomial(1, prob)
        outbreak.append(label)

    df = pd.DataFrame({
        'fever': fever,
        'diarrhea': diarrhea,
        'vomiting': vomiting,
        'water_numeric': water_numeric,
        'outbreak': outbreak
    })
    
    os.makedirs('../data', exist_ok=True)
    df.to_csv(dataset_path, index=False)
    print(f"Dataset generated and saved to {dataset_path}")

print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()

## 2. Exploratory Data Analysis (EDA) & Visualizations

Let's explore the attributes of our data, count class frequencies, and calculate correlation matrices.

In [ ]:
print("--- Data Information ---")
df.info()

print("\n--- Descriptive Statistics ---")
df.describe()

In [ ]:
# Outbreak Label Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='outbreak', data=df, palette='Set2')
plt.title('Outbreak Prevalence in Surveillance Data')
plt.xlabel('Outbreak Occurred (0=No, 1=Yes)')
plt.ylabel('Count')
plt.show()

In [ ]:
# Symptom distributions and relationships with Outbreaks
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
symptoms = ['fever', 'diarrhea', 'vomiting', 'water_numeric']
titles = ['Fever vs Outbreak', 'Diarrhea vs Outbreak', 'Vomiting vs Outbreak', 'Water Contamination vs Outbreak']

for i, sym in enumerate(symptoms):
    ax = axes[i//2, i%2]
    sns.countplot(x=sym, hue='outbreak', data=df, ax=ax, palette='coolwarm')
    ax.set_title(titles[i])
    ax.set_xlabel('Condition Presence (0=Absent, 1=Present)')
    ax.set_ylabel('Report Count')

plt.tight_layout()
plt.show()

### Feature Correlation Heatmap

Let's compute the Pearson correlation coefficient between symptoms, water condition, and the outbreak label. This lets us visualize which inputs are highly associated with outbreaks.

In [ ]:
plt.figure(figsize=(8, 6))
corr = df.corr()
sns.heatmap(corr, annot=True, cmap='RdBu_r', vmin=-1, vmax=1, fmt=".3f", linewidths=0.5)
plt.title('Pearson Correlation Heatmap of Symptoms & Outbreak Risk')
plt.show()

## 3. Model Training & Comparison (Random Forest vs XGBoost)

We split the dataset into training (80%) and testing (20%) sets. Then we build both a `RandomForestClassifier` and an `XGBClassifier` and compare them.

In [ ]:
X = df[['fever', 'diarrhea', 'vomiting', 'water_numeric']]
y = df['outbreak']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

### 3.1 Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)

print(f"Random Forest Accuracy: {rf_acc * 100:.2f}%\n")
print("Classification Report (Random Forest):")
print(classification_report(y_test, rf_pred))

### 3.2 XGBoost Classifier

In [ ]:
# Train an extreme gradient boosting model (XGBoost)
xgb_model = XGBClassifier(
    n_estimators=100, 
    max_depth=3, 
    learning_rate=0.1, 
    random_state=42, 
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_pred)

print(f"XGBoost Accuracy: {xgb_acc * 100:.2f}%\n")
print("Classification Report (XGBoost):")
print(classification_report(y_test, xgb_pred))

### 3.3 Outbreak Confusion Matrix Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# RF Confusion Matrix
rf_cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['No Outbreak', 'Outbreak'], 
            yticklabels=['No Outbreak', 'Outbreak'])
axes[0].set_title('Random Forest Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

# XGB Confusion Matrix
xgb_cm = confusion_matrix(y_test, xgb_pred)
sns.heatmap(xgb_cm, annot=True, fmt='d', cmap='Greens', cbar=False, ax=axes[1],
            xticklabels=['No Outbreak', 'Outbreak'], 
            yticklabels=['No Outbreak', 'Outbreak'])
axes[1].set_title('XGBoost Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.show()

### 3.4 Feature Importance Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
feat_names = X.columns

# RF Importances
rf_importances = rf_model.feature_importances_
rf_indices = np.argsort(rf_importances)[::-1]
sns.barplot(x=rf_importances[rf_indices], y=[feat_names[i] for i in rf_indices], palette='viridis', ax=axes[0])
axes[0].set_title('Random Forest Feature Importances')
axes[0].set_xlabel('Relative Importance')

# XGB Importances
xgb_importances = xgb_model.feature_importances_
xgb_indices = np.argsort(xgb_importances)[::-1]
sns.barplot(x=xgb_importances[xgb_indices], y=[feat_names[i] for i in xgb_indices], palette='mako', ax=axes[1])
axes[1].set_title('XGBoost Feature Importances')
axes[1].set_xlabel('Relative Importance')

plt.tight_layout()
plt.show()

## 4. Exporting the Best Model

We select the **XGBoost Classifier** as our primary model for deployment due to its superior gradient-boosting capabilities and optimization. We serialize it using `pickle` into the Flask backend data folder `../data/model.pkl` so it is loaded at server runtime.

In [ ]:
# Save model path
save_dir = '../data'
os.makedirs(save_dir, exist_ok=True)
model_filepath = os.path.join(save_dir, 'model.pkl')

with open(model_filepath, 'wb') as f:
    pickle.dump(xgb_model, f)
    
print(f"Successfully exported XGBoost model.pkl to {model_filepath}!")